# LusoEnergia — Parte III: Prever o consumo do dia seguinte

**Problema preditivo:** prever o consumo total do **dia seguinte** (kWh/dia, agregação diária de `Global_active_power`).
Esta parte tem duas secções: **3.1** construção de variáveis candidatas (sem fuga de informação) e **3.2** seleção comparada (filter, wrapper, embedded) com validação temporal.

In [2]:
import sys, warnings
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from itertools import combinations

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.feature_selection import SelectKBest, RFE, SelectFromModel, mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit, KFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.exceptions import ConvergenceWarning
import holidays

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

AZUL, LARANJA, TINTA, TINTA_2, GRELHA = "#2a78d6", "#e07b24", "#2b2b28", "#6b6a64", "#e4e3de"
CMAP = LinearSegmentedColormap.from_list("azul", ["#e9f1fc", "#9ec5f4", "#3987e5", "#184f95", "#0d366b"])
SEED = 42

def estilo(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color(GRELHA)
    ax.tick_params(colors=TINTA_2, labelsize=9, length=0)
    ax.grid(axis="y", color=GRELHA, lw=0.8)
    ax.set_axisbelow(True)

# Carregamento dos dados

In [3]:
diario = pd.read_csv("data/lusoenergia_tratado.csv", index_col=0, parse_dates=True).asfreq("D")

# Temperatura horária externa (Sceaux) -- dado de enriquecimento (bónus)
tserie = (pd.read_csv("data/sceaux_temp_horaria_2008_2009.csv", parse_dates=["timestamp"])
          .set_index("timestamp")["temp_c"])
diario["temp"] = tserie.resample("D").mean().reindex(diario.index)

print(f"{diario['total'].notna().sum()} dias válidos | {diario.index.min().date()} -> {diario.index.max().date()}")
diario.head()

548 dias válidos | 2008-05-01 -> 2009-10-31


,total,reativa,intensidade,sub1,sub2,sub3,temp
datetime,,,,,,,
2008-05-01,9.600233,2.511600,1.702361,0.000,0.530,2.586,9.762500
2008-05-02,24.962800,3.426133,4.377778,3.114,0.584,8.687,11.020833
2008-05-03,21.493333,3.366733,3.800972,0.996,0.541,6.535,14.883333
2008-05-04,24.516167,3.490367,4.362500,2.196,2.872,5.464,15.875000
2008-05-05,22.782200,3.095833,3.997778,1.008,0.378,10.197,15.020833


## 3.1 Variáveis candidatas

### Tabela diária de partida

Parte-se da tabela diária guardada no fim da Parte I/II (`data/lusoenergia_diario.csv`), construída a partir dos dados tratados (imputação até 3 h e exclusão dos dias com lacunas longas). Tem o consumo total e as grandezas que o sistema mede, agregadas por dia (kWh/dia). Os dias excluídos na 1.2 estão a `NaN` e as linhas que dependam deles são descartadas mais à frente. Mantêm-se os *outliers*, de acordo com a decisão tomada na Parte I. A tensão não foi guardada (redundante com a potência e a intensidade).

In [ ]:
diario = pd.read_csv("data/lusoenergia_tratado.csv", index_col=0, parse_dates=True).asfreq("D")

# Temperatura horária externa (Sceaux) -- dado de enriquecimento (bónus)
tserie = (pd.read_csv("data/sceaux_temp_horaria_2008_2009.csv", parse_dates=["timestamp"])
          .set_index("timestamp")["temp_c"])
diario["temp"] = tserie.resample("D").mean().reindex(diario.index)

print(f"{diario['total'].notna().sum()} dias válidos | {diario.index.min().date()} -> {diario.index.max().date()}")
diario.head()

### Regra temporal (anti-fuga)

Imagine-se que são as 23:59 do dia *t − 1* e se quer prever o total do dia *t*. Só se pode usar:

1. valores **medidos até ao dia *t − 1*** (inclusive): lags, médias móveis, submedições, tensão, temperatura observada;
2. informação **conhecida com antecedência sobre o dia *t***: calendário (dia da semana, mês, feriado, véspera de feriado).

Nada medido no próprio dia *t* entra nas variáveis, incluindo a temperatura, que só estaria disponível como previsão meteorológica (não observada). Todas as janelas móveis são calculadas sobre a série **deslocada um dia** (`shift(1)`), por isso terminam em *t − 1*. A função seguinte constrói 22 candidatas e a célula a seguir **testa programaticamente** que não há fuga.

In [ ]:
FERIADOS = holidays.country_holidays("FR", years=range(2008, 2011))   # Sceaux fica em França

def construir_features(d):
    # Variáveis para prever o dia t usando apenas informação até t-1 (e calendário de t)
    y = d["total"]
    ys = y.shift(1)                                    # série "vista" ao fim do dia t-1
    X = pd.DataFrame(index=d.index)

    # --- Lags do consumo ---
    for k in (1, 2, 7):
        X[f"lag_{k}"] = y.shift(k)

    # --- Médias / dispersão móveis (janela termina em t-1) ---
    for w in (3, 7, 14):
        X[f"mm_{w}"] = ys.rolling(w, min_periods=int(np.ceil(0.7 * w))).mean()
    X["dp_7"] = ys.rolling(7, min_periods=5).std()
    X["dif_lag1_lag7"] = X["lag_1"] - X["lag_7"]       # variação face à mesma altura da semana anterior

    # --- Calendário (conhecido com antecedência) ---
    dow = d.index.day_of_week
    X["fim_semana"] = (dow >= 5).astype(int)
    X["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    X["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    X["mes_sin"] = np.sin(2 * np.pi * (d.index.month - 1) / 12)
    X["mes_cos"] = np.cos(2 * np.pi * (d.index.month - 1) / 12)
    X["feriado"] = [int(x in FERIADOS) for x in d.index.date]
    X["vespera_feriado"] = [int((x + pd.Timedelta(days=1)) in FERIADOS) for x in d.index]

    # --- Outras grandezas do sistema, do dia anterior ---
    for c in ("sub1", "sub2", "sub3", "outros", "reativa"):
        X[f"{c}_lag1"] = d[c].shift(1)

    # --- Temperatura externa (observada até t-1) ---
    X["temp_lag1"] = d["temp"].shift(1)
    X["temp_mm7"] = d["temp"].shift(1).rolling(7, min_periods=5).mean()
    return X


X_all = construir_features(diario)
print(f"{X_all.shape[1]} variáveis candidatas")

In [ ]:
# Teste de fuga: corromper TUDO o que é medido do dia t em diante não pode alterar as variáveis até ao dia t.
def teste_fuga(d, n=25, seed=0):
    X0 = construir_features(d)
    rng = np.random.default_rng(seed)
    dias = rng.choice(d.index[20:-1], n, replace=False)
    for t in dias:
        d2 = d.copy()
        d2.loc[t:, :] = d2.loc[t:, :] + 1000
        X2 = construir_features(d2)
        assert X0.loc[:t].equals(X2.loc[:t]), f"FUGA detetada em {t.date()}"
    return f"OK: {n} perturbações aleatórias do futuro (a partir do dia t) não alteram as variáveis do dia t."

print(teste_fuga(diario))

In [ ]:
FONTE = {  # variável original (sistema de medição / externa) de que cada candidata depende
    "lag_1": "Global_active_power", "lag_2": "Global_active_power", "lag_7": "Global_active_power",
    "mm_3": "Global_active_power", "mm_7": "Global_active_power", "mm_14": "Global_active_power",
    "dp_7": "Global_active_power", "dif_lag1_lag7": "Global_active_power",
    "fim_semana": "calendário", "dow_sin": "calendário", "dow_cos": "calendário",
    "mes_sin": "calendário", "mes_cos": "calendário", "feriado": "calendário", "vespera_feriado": "calendário",
    "sub1_lag1": "Sub_metering_1", "sub2_lag1": "Sub_metering_2", "sub3_lag1": "Sub_metering_3",
    "outros_lag1": "Global_active_power − submedições", "reativa_lag1": "Global_reactive_power",
"temp_lag1": "temperatura externa", "temp_mm7": "temperatura externa",
}
DESC = {
    "lag_1": "consumo de ontem", "lag_2": "consumo de anteontem", "lag_7": "consumo há 7 dias (mesmo dia da semana)",
    "mm_3": "média dos 3 dias anteriores", "mm_7": "média dos 7 dias anteriores", "mm_14": "média dos 14 dias anteriores",
    "dp_7": "desvio-padrão dos 7 dias anteriores", "dif_lag1_lag7": "lag_1 − lag_7",
    "fim_semana": "dia a prever é sábado/domingo", "dow_sin": "dia da semana (seno)", "dow_cos": "dia da semana (cosseno)",
    "mes_sin": "mês (seno)", "mes_cos": "mês (cosseno)", "feriado": "dia a prever é feriado (FR)",
    "vespera_feriado": "dia a prever é véspera de feriado",
    "sub1_lag1": "cozinha, ontem (kWh)", "sub2_lag1": "lavandaria + frigorífico, ontem (kWh)",
    "sub3_lag1": "aquecedor de água + AC, ontem (kWh)", "outros_lag1": "consumo não submedido, ontem (kWh)",
    "reativa_lag1": "energia reativa, ontem", "temp_lag1": "temperatura média de ontem", "temp_mm7": "temperatura média dos 7 dias anteriores",
}
assert set(FONTE) == set(X_all.columns) == set(DESC)

tab_feat = pd.DataFrame({"descrição": DESC, "fonte": FONTE})
tab_feat["usa dados até"] = ["t − 1" if f != "calendário" else "calendário de t (conhecido)" for f in tab_feat["fonte"]]
tab_feat

### Conjunto de dados final e reserva temporal de teste

Descartam-se os dias sem alvo ou sem alguma variável (início da série, que não tem 14 dias de história, e dias vizinhos das lacunas). Os **últimos 20 % dos dias** ficam reservados como **teste final** e não são usados em nenhuma escolha (seleção, hiperparâmetros). Toda a validação cruzada da 3.2 é feita só dentro do treino.

In [ ]:
dados = X_all.join(diario["total"].rename("y")).dropna()
FEATS = list(X_all.columns)

corte = int(len(dados) * 0.8)
treino, teste = dados.iloc[:corte], dados.iloc[corte:]
Xtr, ytr = treino[FEATS], treino["y"]
Xte, yte = teste[FEATS], teste["y"]

print(f"{len(dados)} dias utilizáveis")
print(f"treino: {treino.index.min().date()} -> {treino.index.max().date()} ({len(treino)} dias)")
print(f"teste : {teste.index.min().date()} -> {teste.index.max().date()} ({len(teste)} dias)")

In [ ]:
# Correlação de Spearman de cada candidata com o consumo do dia seguinte (só treino)
rho = Xtr.apply(lambda c: c.corr(ytr, method="spearman")).sort_values()
fig, ax = plt.subplots(figsize=(7, 6.2))
ax.barh(rho.index, rho.values, color=[AZUL if v >= 0 else LARANJA for v in rho.values])
ax.set_xlabel("correlação de Spearman com o consumo do dia", fontsize=9, color=TINTA_2)
ax.set_title("Variáveis candidatas vs. alvo (treino)", loc="left", fontsize=11, fontweight="bold", color=TINTA)
estilo(ax); ax.grid(axis="x", color=GRELHA, lw=0.8); ax.grid(axis="y", visible=False)
plt.show()

## 3.2 Seleção de variáveis comparada

### Porque não KFold aleatório

Num problema temporal, as linhas não são independentes: o consumo de um dia é muito parecido com o dos vizinhos (autocorrelação) e as nossas variáveis (lags, médias móveis) **reutilizam o mesmo consumo em várias linhas**. Com `KFold(shuffle=True)`:

- o modelo treina com dias **posteriores** ao dia que vai ser validado, ou seja, usa o futuro para prever o passado;
- dias consecutivos, quase iguais e com janelas sobrepostas, ficam uns no treino e outros na validação, de modo que o erro de validação é otimista (*leakage*).

O `TimeSeriesSplit` valida sempre num bloco **posterior** ao treino (janela expansiva), que é o que acontece em produção. Para o ver, compara-se o mesmo modelo (regressão linear com todas as variáveis) com os dois esquemas, só no treino:

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
base = Pipeline([("sc", StandardScaler()), ("m", LinearRegression())])

mae_kf = -cross_val_score(base, Xtr, ytr, cv=KFold(5, shuffle=True, random_state=SEED), scoring="neg_mean_absolute_error")
mae_ts = -cross_val_score(base, Xtr, ytr, cv=tscv, scoring="neg_mean_absolute_error")
pd.DataFrame({"KFold aleatório": [mae_kf.mean()], "TimeSeriesSplit": [mae_ts.mean()]},
             index=["MAE médio (kWh/dia)"]).round(3)

### Os três métodos, dentro de um `Pipeline`

Cada método é um `Pipeline` `StandardScaler → seletor → LinearRegression`. Como o seletor está **dentro** do pipeline, em cada fold da validação temporal o escalonamento e a seleção são reajustados **só com o treino desse fold**. Se a seleção fosse feita antes, com todos os dados, a informação do fold de validação já teria influenciado as variáveis escolhidas.

| Tipo | Método | Hiperparâmetro (escolhido por `GridSearchCV` com `TimeSeriesSplit`) |
|---|---|---|
| Filter | informação mútua com o alvo (`SelectKBest`) | nº de variáveis `k` |
| Wrapper | RFE sobre regressão linear | nº de variáveis |
| Embedded | Lasso (`SelectFromModel`, coeficientes ≠ 0) | força da penalização `α` |

In [ ]:
KS = [3, 5, 8, 12, 16]
MI = lambda X, y: mutual_info_regression(X, y, random_state=SEED)

def pipe(sel):
    return Pipeline([("sc", StandardScaler()), ("sel", sel), ("m", LinearRegression())])

CONFIG = {
    "filter (MI)":     (pipe(SelectKBest(MI)),                              {"sel__k": KS}),
    "wrapper (RFE)":   (pipe(RFE(LinearRegression())),                       {"sel__n_features_to_select": KS}),
    "embedded (Lasso)": (pipe(SelectFromModel(Lasso(max_iter=50000), threshold=1e-6)),
                         {"sel__estimator__alpha": [float(a) for a in np.round(np.logspace(-2.5, -0.3, 6), 4)]}),
}

pesquisas = {}
for nome, (p, grelha) in CONFIG.items():
    gs = GridSearchCV(p, grelha, cv=tscv, scoring="neg_mean_absolute_error", refit=True)
    gs.fit(Xtr, ytr)
    pesquisas[nome] = gs
    print(f"{nome:18s} melhor: {gs.best_params_} | MAE CV = {-gs.best_score_:.3f}")

### Subconjuntos selecionados

Variáveis escolhidas por cada método, ajustado a todo o treino com os melhores hiperparâmetros:

In [ ]:
final = pd.DataFrame({n: gs.best_estimator_.named_steps["sel"].get_support() for n, gs in pesquisas.items()},
                     index=FEATS)
final["nº métodos"] = final.sum(axis=1)
final = final.sort_values("nº métodos", ascending=False)

vis = final.copy()
for c in vis.columns[:-1]:
    vis[c] = vis[c].map({True: "✓", False: ""})
display(vis)

sel = {n: set(final.index[final[n]]) for n in CONFIG}
print({n: len(s) for n, s in sel.items()}, "variáveis selecionadas")
print("Comuns aos 3 métodos:", sorted(set.intersection(*sel.values())))
print("Selecionadas por pelo menos 1:", len(set.union(*sel.values())), "de", len(FEATS))

### Regressão linear: todas vs. selecionadas

Compara-se a regressão linear com **todas** as variáveis e com cada subconjunto, e dois modelos *naive* de referência (repetir o consumo de ontem ou o de há 7 dias). O MAE de validação cruzada é calculado no treino; o MAE do **teste** é o único valor que ainda não influenciou nenhuma decisão.

In [ ]:
def metricas(pred):
    return mean_absolute_error(yte, pred), np.sqrt(mean_squared_error(yte, pred))

linhas = {}
# todas as variáveis
m_todas = clone(base).fit(Xtr, ytr)
linhas[f"todas ({len(FEATS)})"] = {"nº variáveis": len(FEATS),
                        "MAE CV treino": -cross_val_score(base, Xtr, ytr, cv=tscv, scoring="neg_mean_absolute_error").mean(),
                        **dict(zip(["MAE teste", "RMSE teste"], metricas(m_todas.predict(Xte))))}
# subconjuntos
for n, gs in pesquisas.items():
    k = int(gs.best_estimator_.named_steps["sel"].get_support().sum())
    linhas[n] = {"nº variáveis": k, "MAE CV treino": -gs.best_score_,
                 **dict(zip(["MAE teste", "RMSE teste"], metricas(gs.predict(Xte))))}
# referências naive
for nome, col in [("naive: ontem", "lag_1"), ("naive: há 7 dias", "lag_7")]:
    mae_cv = np.mean([mean_absolute_error(ytr.iloc[v], Xtr[col].iloc[v]) for _, v in tscv.split(Xtr)])
    linhas[nome] = {"nº variáveis": 0, "MAE CV treino": mae_cv, **dict(zip(["MAE teste", "RMSE teste"], metricas(Xte[col])))}

comparacao = pd.DataFrame(linhas).T
comparacao["nº variáveis"] = comparacao["nº variáveis"].astype(int)
display(comparacao.round(3))

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(comparacao)); w = 0.38
ax.bar(x - w / 2, comparacao["MAE CV treino"], w, color=AZUL, label="MAE CV (treino)")
ax.bar(x + w / 2, comparacao["MAE teste"], w, color=LARANJA, label="MAE teste")
ax.set_xticks(x, comparacao.index, rotation=25, ha="right")
ax.set_ylabel("MAE (kWh/dia)", fontsize=9, color=TINTA_2)
ax.legend(frameon=False, fontsize=9)
ax.set_title("Erro por conjunto de variáveis", loc="left", fontsize=11, fontweight="bold", color=TINTA)
estilo(ax); plt.show()

### Estabilidade da seleção entre folds

Para cada método, repete-se a seleção (com os hiperparâmetros escolhidos) no treino de cada fold do `TimeSeriesSplit` e regista-se que variáveis ficam. Calcula-se:

- a **frequência de seleção** de cada variável (fração dos folds em que foi escolhida);
- a **similaridade de Jaccard** média entre pares de folds (1 = sempre o mesmo subconjunto; 0 = nada em comum).

In [ ]:
def mascaras_por_fold(gs):
    est = gs.best_estimator_
    linhas = []
    for tr, _ in tscv.split(Xtr):
        e = clone(est)
        Z = e.named_steps["sc"].fit_transform(Xtr.iloc[tr])      # só escalonador + seletor (a regressão não interessa aqui)
        e.named_steps["sel"].fit(Z, ytr.iloc[tr])
        linhas.append(e.named_steps["sel"].get_support())
    return pd.DataFrame(linhas, columns=FEATS, index=[f"fold {i+1}" for i in range(len(linhas))])

folds = {n: mascaras_por_fold(gs) for n, gs in pesquisas.items()}
freq = pd.DataFrame({n: m.mean() for n, m in folds.items()})

def jaccard_medio(m):
    s = [set(m.columns[r]) for r in m.values]
    return np.mean([len(a & b) / len(a | b) for a, b in combinations(s, 2)])

estab = pd.DataFrame({
    "nº médio de variáveis": {n: m.sum(axis=1).mean() for n, m in folds.items()},
    "Jaccard médio entre folds": {n: jaccard_medio(m) for n, m in folds.items()},
    "variáveis em 100 % dos folds": {n: int((m.mean() == 1).sum()) for n, m in folds.items()},
})
display(estab.round(2))

ordem = freq.sum(axis=1).sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(6.2, 0.33 * len(ordem) + 1.6))
ax.imshow(freq.loc[ordem].values, cmap=CMAP, vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(freq.shape[1]), freq.columns); ax.set_yticks(range(len(ordem)), ordem)
ax.xaxis.tick_top()
for i, f in enumerate(ordem):
    for j, n in enumerate(freq.columns):
        v = freq.loc[f, n]
        ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=8, color="white" if v > 0.5 else TINTA)
ax.tick_params(length=0, labelsize=9)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_title("Frequência de seleção por fold", loc="left", fontsize=11, fontweight="bold", color=TINTA, pad=28)
plt.show()

### Resumo para a recomendação

Agrega-se por **variável original do sistema**: o que interessa à LusoEnergia é o que deve **medir e manter**, e não cada lag individual. Para cada fonte, mostra-se a maior frequência de seleção (entre folds e métodos) das candidatas que dela dependem.

In [ ]:
fonte_freq = freq.groupby(FONTE).max()
fonte_freq["média dos 3 métodos"] = fonte_freq.mean(axis=1)
fonte_freq = fonte_freq.sort_values("média dos 3 métodos", ascending=False)
display((fonte_freq * 100).round(0).astype(int).astype(str) + " %")

nucleo = final.index[final["nº métodos"] == 3].tolist()
estaveis = freq.index[(freq >= 0.8).sum(axis=1) >= 2].tolist()
print("Escolhidas pelos 3 métodos (treino completo):", nucleo)
print("Escolhidas em ≥ 80 % dos folds por pelo menos 2 métodos:", estaveis)
print("Nunca escolhidas por nenhum método:", final.index[final["nº métodos"] == 0].tolist())

### Conclusões

*(Completar com os números que a execução produzir. Estrutura sugerida:)*

1. **Subconjuntos:** quantas variáveis cada método escolheu, qual a interseção entre os três e onde divergem (por exemplo, o filter por informação mútua trata cada variável isoladamente e tende a escolher várias variáveis redundantes, como `lag_1`, `mm_3` e `mm_7`, que contêm informação quase igual; o Lasso e o RFE, ao olharem para o conjunto, tendem a manter só uma delas).
2. **Todas vs. selecionadas:** o MAE (CV e teste) piora, melhora ou fica igual ao reduzir de 22 para *k* variáveis? Se fica igual com menos variáveis, mantém-se o modelo mais simples e mais barato de operar. Comparar também com os modelos *naive*.
3. **Estabilidade:** Jaccard e frequências. Uma variável escolhida em todos os folds é fiável; uma que aparece e desaparece depende da amostra (colinearidade).
4. **Recomendação à LusoEnergia:**
   - **Medir e manter** as fontes com maior frequência de seleção agregada (provavelmente o consumo global histórico e o calendário, que não custam nada; verificar o papel das submedições e da temperatura);
   - **Poderá dispensar** as fontes que nunca ou quase nunca são escolhidas (por exemplo `reativa`, se for o caso), ou mantê-las apenas se servirem outro objetivo (qualidade da rede);
   - Se alguma submedição for selecionada de forma estável, justifica-se manter o sensor correspondente; se não, o consumo global chega para este objetivo de previsão.
5. **Limitações:** só ~1,5 ano de dados (um ciclo anual completo e meio), a temperatura de dia *t* não foi usada (numa implementação real poder-se-ia usar a previsão meteorológica), e a seleção em dados colineares é sensível à amostra.